# Does dirty air fill hospital wards? A regional analysis of UK air pollution and respiratory admissions.

This notebook explores the relationship between air pollution (PM2.5) and respiratory hospital admissions across UK regions.
We merge DEFRA's UK air quality monitoring data with NHS respiratory admissions data and run an OLS regression to show how pollution levels predict hospital demand.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
import warnings
warnings.filterwarnings('ignore')

# Set a beautiful modern theme for all plots
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Load the merged dataset
data_path = os.path.join("data", "clean", "merged_regional_data.csv")
df = pd.read_csv(data_path)
df.head()

## 2. Choropleth Map of Average PM2.5 by Region

In [ ]:
# Load local English regions GeoJSON
regions_geojson_path = 'eer.json'
gdf = gpd.read_file(regions_geojson_path)

# Calculate average PM2.5 across all years per region
avg_pm25 = df.groupby('region')['PM2.5'].mean().reset_index()

rename_map = {
    'London': 'London',
    'North West': 'North West',
    'West Midlands': 'Midlands',
    'East Midlands': 'Midlands',
    'East of England': 'East of England',
    'South East': 'South East',
    'South West': 'South West',
    'North East': 'North East and Yorkshire',
    'Yorkshire and The Humber': 'North East and Yorkshire'
}

gdf['mapped_region'] = gdf['EER13NM'].map(rename_map)
gdf_merged = gdf.merge(avg_pm25, left_on='mapped_region', right_on='region', how='left')

# Plot setup
fig, ax = plt.subplots(1, 1, figsize=(10, 8), dpi=100)
gdf_merged.plot(column='PM2.5', ax=ax, legend=True,
                legend_kwds={'label': "Average PM2.5 (μg/m³)", 'orientation': "horizontal", 'shrink': 0.6},
                cmap='YlOrBr', edgecolor='gray', linewidth=0.5, missing_kwds={'color': 'whitesmoke'})

ax.set_title('Average PM2.5 by Region (2015-2023)\nSpatial distribution of air pollution across England', loc='left', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Time Series of Pollution Trends (2015-2023)

In [ ]:
plt.figure(figsize=(12, 6), dpi=100)
ax = sns.lineplot(data=df, x='year', y='PM2.5', hue='region', 
                  marker='o', markersize=8, linewidth=2.5, palette='husl')

plt.title('PM2.5 Trends Over Time by Region', loc='left', pad=15)
plt.ylabel('PM2.5 (μg/m³)')
plt.xlabel('Year')
sns.despine(left=True, bottom=True)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title='Region')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Scatter Plot: PM2.5 vs Respiratory Admissions

In [ ]:
plt.figure(figsize=(10, 6), dpi=100)

# Scatter points
sns.scatterplot(data=df, x='PM2.5', y='admission_rate_per_100k', 
                hue='region', s=120, alpha=0.8, edgecolor='w', linewidths=0.8, palette='husl')

# Regression line
sns.regplot(data=df, x='PM2.5', y='admission_rate_per_100k', 
            scatter=False, color='crimson', line_kws={"linestyle": "--", "linewidth": 2})

plt.title('PM2.5 vs Respiratory Admission Rate', loc='left', pad=15)
plt.xlabel('PM2.5 (μg/m³)')
plt.ylabel('Respiratory Admission Rate (per 100k)')
sns.despine(left=True, bottom=True)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title='Region')
plt.tight_layout()
plt.show()

## 5. Expanded Regression Analysis

We started with a simple relationship. Now, we'll build a series of OLS regression models to better isolate the impact of PM2.5:
1. **Baseline Model:** Only PM2.5.
2. **Pollutant Controls:** Adding NO2 and O3.
3. **Fixed Effects:** Controlling for unobserved differences across `region` and `year`.
4. **Interaction Model:** Exploring if the combined effect of PM2.5 and NO2 is worse than their individual effects.

In [ ]:
# 1. Baseline Model
mod1 = smf.ols('admission_rate_per_100k ~ Q("PM2.5")', data=df).fit()

# 2. Pollutant Controls
mod2 = smf.ols('admission_rate_per_100k ~ Q("PM2.5") + NO2 + O3', data=df).fit()

# 3. Fixed Effects (Region and Year)
mod3 = smf.ols('admission_rate_per_100k ~ Q("PM2.5") + NO2 + O3 + C(region) + C(year)', data=df).fit()

# 4. Interaction Term
mod4 = smf.ols('admission_rate_per_100k ~ Q("PM2.5") * NO2 + O3 + C(region) + C(year)', data=df).fit()

from statsmodels.iolib.summary2 import summary_col
print(summary_col([mod1, mod2, mod3, mod4], 
                  model_names=['Baseline', 'Pollutants', 'Fixed Effects', 'Interaction'],
                  stars=True, float_format='%0.3f',
                  info_dict={'R-squared': lambda x: f"{x.rsquared:0.3f}",
                             'No. Observations': lambda x: f"{int(x.nobs)}"}))

# We'll use the fixed effects model (mod3) for our final predictions and validation.
best_model = mod3

### Visualizing the Pollution Effect
To make the impact easier to digest, let's plot the coefficients with their confidence intervals from the Fixed Effects model.

In [ ]:
# Extract coefficients and conf intervals for the pollutants from Model 3
params = mod3.params[['Q("PM2.5")', 'NO2', 'O3']]
conf = mod3.conf_int().loc[['Q("PM2.5")', 'NO2', 'O3']]
conf['error'] = conf[1] - params

plt.figure(figsize=(8, 5), dpi=100)
plt.errorbar(x=params.index, y=params.values, yerr=conf['error'], fmt='o', 
             color='crimson', markersize=10, linewidth=2, capsize=5)

plt.title('Impact of Pollutants on Respiratory Admissions (per 100k)\nControlling for Region and Year', loc='left', pad=15)
plt.ylabel('Coefficient (Impact)')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
sns.despine()
plt.tight_layout()
plt.show()

## 6. Residuals Plot for Final Model Validation

In [ ]:
# Calculate predictions and residuals for the Fixed Effects model
df['predicted'] = best_model.fittedvalues
df['residuals'] = best_model.resid

plt.figure(figsize=(10, 6), dpi=100)
sns.residplot(x=df['predicted'], y=df['residuals'], lowess=True, 
              scatter_kws={'alpha': 0.7, 's': 80, 'edgecolor': 'w', 'linewidths': 0.5, 'color': 'steelblue'},
              line_kws={'color': 'crimson', 'linewidth': 2})

plt.title('Residuals vs Predicted Values (Fixed Effects Model)', loc='left', pad=15)
plt.xlabel('Predicted Admission Rate (per 100k)')
plt.ylabel('Residuals')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()